In [ ]:
import os
from dotenv import load_dotenv
import redcap
import pandas as pd
import pandasql as psql
from datetime import datetime, timedelta
import re
from sklearn.metrics.pairwise import cosine_similarity

# Load environment variables
load_dotenv()

# Initialize REDCap projects
df = redcap.Project(
    os.getenv('REDCAP_MAIN_URL'),
    os.getenv('REDCAP_MAIN_TOKEN')
)

ck_wk = redcap.Project(
    os.getenv('REDCAP_CK_WK_URL'),
    os.getenv('REDCAP_CK_WK_TOKEN')
)

indigo = redcap.Project(
    os.getenv('REDCAP_INDIGO_URL'),
    os.getenv('REDCAP_INDIGO_TOKEN')
)

In [71]:
consent=df.export_records(forms=['consent'])
consent=pd.DataFrame(consent)

In [72]:
consent=consent[['participant_id','ck_wkno','consent_complete']]


In [73]:
consent=consent.rename(columns={ 'ck_wkno':'wk_ckno'})



In [74]:
consent=consent[
    (consent['consent_complete']=='2')
]

In [75]:
senlog=ck_wk.export_records(forms=['enumeration_and_sensitisation'])
senlog=pd.DataFrame(senlog)

In [76]:
senlog=senlog[
    (senlog['wk_ckno']!='')
]

In [77]:
data_mergh=pd.merge(consent,senlog,on='wk_ckno',how='inner')

In [78]:
columns_to_check='enu_name','enu_mname','enu_fname'

In [79]:
simiratues=data_mergh[data_mergh.duplicated(subset=columns_to_check,keep=False)]

In [83]:
simiratues.sort_values('enu_name').to_csv('simiratues.csv',index=False)
